In [1]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import root_mean_squared_error
import ast
import numpy as np
import pandas as pd
from shapely.geometry import box
import geopandas as gpd
import matplotlib.pyplot as pyplot
from matplotlib.lines import Line2D

# Collect all data in a dictionary

In [2]:
folder = Path("csv/SCFG2x2_degree/")

lac = pd.DataFrame()
gac = pd.DataFrame()

for file in folder.rglob("*.csv"):
    df = pd.read_csv(file)
    lac = pd.concat([lac, df[df["datatype"] == "lac"]])
    gac = pd.concat([gac, df[df["datatype"] == "gac"]])

all_regions = set(lac["region"]).union(gac["region"])

regions = {
    region: {
        "lac": lac[lac["region"] == region],
        "gac": gac[gac["region"] == region]
    }
    for region in all_regions
}

print(lac.head())
print(gac.head())

# Example: access one region
print(regions["San_Rafael_South_America"]["lac"])
print(regions["San_Rafael_South_America"]["gac"])

        date datatype satellite       region  \
91  19920401      lac    noaa11  rus_Siberia   
93  19920402      lac    noaa11  rus_Siberia   
95  19920403      lac    noaa11  rus_Siberia   
97  19920404      lac    noaa11  rus_Siberia   
99  19920405      lac    noaa11  rus_Siberia   

                                                 data  
91  [nan, nan, nan, nan, nan, 99.99532318115234, n...  
93  [nan, 99.90837860107422, nan, nan, nan, nan, n...  
95  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
97  [nan, 99.76417541503906, nan, nan, nan, nan, n...  
99  [nan, nan, nan, nan, nan, nan, nan, nan, 99.71...  
       date datatype satellite       region                 data
0  19920101      gac    noaa11  rus_Siberia                [nan]
1  19920102      gac    noaa11  rus_Siberia  [99.92376708984375]
2  19920103      gac    noaa11  rus_Siberia  [99.85140991210938]
3  19920104      gac    noaa11  rus_Siberia  [99.95785522460938]
4  19920105      gac    noaa11  rus_Siberia       

KeyError: 'San_Rafael_South_America'

# Calculate mean, max, min, sd, rmse

In [3]:
summary_rows = []

for data_df in [lac, gac]:

    for (region, satellite, datatype), group in data_df.groupby(
        ["region", "satellite", "datatype"]
    ):

        values = []

        for data_string in group["data"]:

            nums = [
                np.nan if x.strip() == "nan" else float(x)
                for x in data_string.strip("[]").split(",")
                if x.strip() != ""
            ]

            values.extend(nums)

        values = np.array(values, dtype=float)

        # remove NaNs
        values = values[~np.isnan(values)]

        summary_rows.append({
            "region": region,
            "satellite": satellite,
            "datatype": datatype,
            "mean": np.mean(values) if len(values) else np.nan,
            "std": np.std(values) if len(values) else np.nan,
            "count": len(values)
        })


summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["region", "satellite", "datatype"]
    )
    .reset_index(drop=True)
)



# RMSE calculation
# Compare only dates where both LAC and GAC have values


rmse_rows = []


for (region, satellite), lac_group in lac.groupby(
    ["region", "satellite"]
):

    # matching GAC
    gac_group = gac[
        (gac["region"] == region) &
        (gac["satellite"] == satellite)
    ]


    # group by date because there are multiple rows per day
    lac_dates = lac_group.groupby("date")
    gac_dates = gac_group.groupby("date")


    daily_rmse = []


    # only use dates existing in both datasets
    common_dates = set(lac_dates.groups.keys()).intersection(
        set(gac_dates.groups.keys())
    )


    for date in sorted(common_dates):

        lac_day = lac_dates.get_group(date)
        gac_day = gac_dates.get_group(date)


        # collect all LAC values for this date
        lac_values = []

        for s in lac_day["data"]:

            vals = [
                np.nan if x.strip() == "nan" else float(x)
                for x in s.strip("[]").split(",")
                if x.strip() != ""
            ]

            lac_values.extend(vals)


        lac_values = np.array(
            lac_values,
            dtype=float
        )

        lac_values = lac_values[
            ~np.isnan(lac_values)
        ]


        # collect GAC values for this date
        gac_values = []

        for s in gac_day["data"]:

            vals = [
                np.nan if x.strip() == "nan" else float(x)
                for x in s.strip("[]").split(",")
                if x.strip() != ""
            ]

            gac_values.extend(vals)


        gac_values = np.array(
            gac_values,
            dtype=float
        )

        gac_values = gac_values[
            ~np.isnan(gac_values)
        ]


        # skip date if either has no values
        if len(lac_values) == 0 or len(gac_values) == 0:
            continue


        # GAC is one value per date
        gac_value = gac_values[0]


        rmse = root_mean_squared_error(
            np.full(len(lac_values), gac_value),
            lac_values
        )


        daily_rmse.append(rmse)



    rmse_rows.append({

        "region": region,
        "satellite": satellite,

        # mean RMSE over all valid dates
        "rmse": np.mean(daily_rmse)
            if len(daily_rmse) > 0
            else np.nan,

        "days_used": len(daily_rmse)

    })



rmse_summary = (
    pd.DataFrame(rmse_rows)
    .sort_values(
        ["region", "satellite"]
    )
    .reset_index(drop=True)
)




# Display


print("Statistics:")
print(summary)


print("\nRMSE:")
print(rmse_summary)




# Save CSV files


summary.to_csv(
    "results/summary_statistics_SCFG2x2_degree.csv",
    index=False
)


rmse_summary.to_csv(
    "results/rmse_summary_SCFG2x2_degree.csv",
    index=False
)


print("\nSaved:")
print("results/summary_statistics_SCFG2x2_degree.csv")
print("results/rmse_summary_SCFG2x2_degree.csv")

Statistics:
           region satellite datatype       mean        std  count
0   can_Auyuittuq    noaa11      gac  83.576376  31.312830    896
1   can_Auyuittuq    noaa11      lac  78.202157  33.255914   2104
2   can_Auyuittuq    noaa14      gac  81.176006  33.277042    868
3   can_Auyuittuq    noaa14      lac  80.104266  33.301921   2488
4      chl_Rafael    noaa11      gac   9.220362  14.973420   2758
5      chl_Rafael    noaa11      lac  25.703965  27.988849   2295
6      chl_Rafael    noaa14      gac   8.029991  10.769084   3218
7      chl_Rafael    noaa14      lac  35.162601  31.377390   2836
8   nor_Jotunheim    noaa11      gac  50.802002  39.934682   1168
9   nor_Jotunheim    noaa11      lac  45.102941  37.898564   2836
10  nor_Jotunheim    noaa14      gac  55.062657  40.584599   1168
11  nor_Jotunheim    noaa14      lac  56.945966  38.589704   3506
12    rus_Karelia    noaa11      gac  42.766000  44.721701   1144
13    rus_Karelia    noaa11      lac  29.966526  40.922970   244